# 04 — Label Creation + Leakage Audit + Group Splits

Purpose:
1. create the binary ML target;
2. audit whether the synthetic generator leaks the label into activity features;
3. create one persistent group-aware train/validation/test split used by every model notebook.

**Important:** if the leakage audit shows that `usage_count` nearly determines the label, regenerate the synthetic data rather than hiding the problem with a more complex model.

In [1]:
# AI-Based IAM Permission Optimizer — Model V2
# Run notebooks in order: 01 → 10
# Raw CSVs should be available in the project root or adjust RAW_DIR below.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_DIR / "engineered_dataset.csv")

valid_labels = {"INTENDED", "EXCESSIVE"}
unexpected = set(df["permission_status"].dropna().unique()) - valid_labels
if unexpected:
    raise ValueError(f"Unexpected labels: {unexpected}")

df["target"] = df["permission_status"].eq("EXCESSIVE").astype(int)

print(df["target"].value_counts())
print("EXCESSIVE rate:", round(df["target"].mean(), 4))

target
0    16000
1     4000
Name: count, dtype: int64
EXCESSIVE rate: 0.2


In [2]:
# Label-generation audit: can a tiny model using only usage_count reproduce the labels?
leakage_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
usage_only_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("tree", DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42))
])

usage_only_f1 = cross_val_score(
    usage_only_model,
    df[["usage_count"]],
    df["target"],
    groups=df["user_id"],
    cv=leakage_cv,
    scoring="f1",
    n_jobs=1
)

print("Usage-count-only grouped CV F1 by fold:", np.round(usage_only_f1, 4))
print("Mean F1:", round(usage_only_f1.mean(), 4))

if usage_only_f1.mean() >= 0.90:
    print("\nWARNING: target is highly predictable from usage_count alone.")
    print("Regenerate synthetic labels independently from simulated activity before claiming real-world generalization.")

Usage-count-only grouped CV F1 by fold: [0.501  0.4927 0.5035 0.5086 0.4878]
Mean F1: 0.4987


In [3]:
# Check whether role/action pairs have conflicting labels.
role_action_labels = df.groupby(["role_id", "action"])["target"].nunique()
conflicting = role_action_labels[role_action_labels > 1]
print("Conflicting role/action labels:", len(conflicting))

Conflicting role/action labels: 0


In [4]:
# Persistent group-aware split. Same user can never appear in different splits.
X_split = df.drop(columns=["target", "permission_status"], errors="ignore")
y_split = df["target"]
groups = df["user_id"]

outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_pool_idx, test_idx = next(outer.split(X_split, y_split, groups=groups))

inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=43)
fit_idx, val_idx = next(inner.split(
    X_split.iloc[train_pool_idx],
    y_split.iloc[train_pool_idx],
    groups=groups.iloc[train_pool_idx]
))

# Convert inner indices back to original row positions.
fit_idx_global = np.asarray(train_pool_idx)[fit_idx]
val_idx_global = np.asarray(train_pool_idx)[val_idx]

split = np.full(len(df), "unused", dtype=object)
split[fit_idx_global] = "train"
split[val_idx_global] = "validation"
split[test_idx] = "test"

df["split"] = split

if (df["split"] == "unused").any():
    raise RuntimeError("Some rows were not assigned a split.")

train_users = set(df.loc[df["split"] == "train", "user_id"])
val_users = set(df.loc[df["split"] == "validation", "user_id"])
test_users = set(df.loc[df["split"] == "test", "user_id"])

print("Train rows:", int((df["split"] == "train").sum()))
print("Validation rows:", int((df["split"] == "validation").sum()))
print("Test rows:", int((df["split"] == "test").sum()))
print("Train users:", len(train_users), "Validation users:", len(val_users), "Test users:", len(test_users))
print("Train/Test overlap:", train_users & test_users)
print("Validation/Test overlap:", val_users & test_users)

Train rows: 12000
Validation rows: 4000
Test rows: 4000
Train users: 60 Validation users: 20 Test users: 20
Train/Test overlap: set()
Validation/Test overlap: set()


In [5]:
# Save labeled dataset and persistent split manifest.
labeled_path = DATA_DIR / "labeled_dataset.csv"
split_manifest_path = DATA_DIR / "splits.csv"

df.to_csv(labeled_path, index=False)
df[["row_id", "user_id", "split"]].to_csv(split_manifest_path, index=False)

meta = {
    "usage_only_grouped_cv_f1_mean": float(usage_only_f1.mean()),
    "train_users": len(train_users),
    "validation_users": len(val_users),
    "test_users": len(test_users),
}
with open(ARTIFACT_DIR / "split_metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Saved:", labeled_path)
print("Saved:", split_manifest_path)

Saved: C:\Users\LENOVO\Downloads\IAM_Model_V2_Notebooks\data\labeled_dataset.csv
Saved: C:\Users\LENOVO\Downloads\IAM_Model_V2_Notebooks\data\splits.csv
